# EDA

Analizaremos el dataset_caracteristicas_Train_V1_ALL

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configurar el estilo visual de Seaborn para que los gráficos luzcan profesionales
sns.set_theme(style="whitegrid", palette="muted")

# 1. Cargar el dataset consolidado
df = pd.read_csv('dataset_caracteristicas_train_V1_ALL_OriginIncluded.csv')

# 2. Verificamos que todo esté en orden
print(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.")
print("\nMuestra de las primeras filas:")
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '../Obtencion_Metricas/dataset_caracteristicas_train_V1_ALL_OriginIncluded.csv'

In [ ]:
# Crear una figura
plt.figure(figsize=(8, 5))

# Gráfico de barras contando cuántos hay de cada uno
ax = sns.countplot(data=df, x='label', hue='label', palette={'bonafide': '#2ca02c', 'spoof': '#d62728'}, legend=False)

plt.title('Balance de Clases: Audios Reales vs Deepfakes', fontsize=14, fontweight='bold')
plt.xlabel('Clase de Audio', fontsize=12)
plt.ylabel('Cantidad de Archivos', fontsize=12)

# Añadir los números exactos encima de las barras
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=11, color='black', xytext=(0, 5),
                textcoords='offset points')

plt.show()

In [ ]:
# Selecciona 2 o 3 métricas de tu dataset para analizar
metricas_a_analizar = ['nombre_de_tu_caracteristica_1', 'nombre_de_tu_caracteristica_2'] 

plt.figure(figsize=(14, 5))

for i, metrica in enumerate(metricas_a_analizar, 1):
    plt.subplot(1, len(metricas_a_analizar), i)
    
    # KDE Plot: Dibuja las curvas de distribución
    sns.kdeplot(data=df, x=metrica, hue='label', fill=True, 
                palette={'bonafide': '#2ca02c', 'spoof': '#d62728'}, alpha=0.5)
    
    plt.title(f'Distribución de: {metrica}', fontsize=12)
    plt.xlabel('Valor de la métrica')
    plt.ylabel('Densidad')

plt.tight_layout()
plt.show()

# CONCLUSIÓN ESPERADA: 
# Si las curvas roja y verde están muy separadas, ¡esa métrica es oro! 
# Si están completamente superpuestas, esa métrica no ayudará mucho al modelo.

In [ ]:
# Pon aquí el nombre de tu mejor métrica numérica
mejor_metrica = 'nombre_de_tu_caracteristica_1'

plt.figure(figsize=(12, 6))

# Ordenar los generadores para que el gráfico tenga sentido
orden_generadores = sorted(df['modelo_ia_origen'].unique())

sns.boxplot(data=df, x='modelo_ia_origen', y=mejor_metrica, order=orden_generadores, hue='modelo_ia_origen', palette='viridis', legend=False)

plt.title(f'Comportamiento de "{mejor_metrica}" según el Generador de IA', fontsize=14, fontweight='bold')
plt.xlabel('Modelo Generador ( - es Real)', fontsize=12)
plt.ylabel(f'Valor de {mejor_metrica}', fontsize=12)

# Añadir una línea roja punteada que marque la media de los audios reales para comparar fácilmente
media_real = df[df['modelo_ia_origen'] == '-'][mejor_metrica].mean()
plt.axhline(media_real, color='red', linestyle='--', linewidth=2, label='Media Audios Reales')
plt.legend()

plt.show()

In [ ]:
# Seleccionar solo las columnas numéricas (excluimos textos y etiquetas)
columnas_numericas = df.select_dtypes(include=['float64', 'int64']).columns

# Si tienes 100 características, el gráfico será ilegible. 
# Tomamos solo las primeras 15 para visualizar (puedes ajustar este número)
columnas_a_correlacionar = columnas_numericas[:15]

# Calcular la matriz de correlación
matriz_corr = df[columnas_a_correlacionar].corr()

plt.figure(figsize=(12, 10))

# Crear el mapa de calor
sns.heatmap(matriz_corr, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=.5, cbar_kws={"shrink": .8})

plt.title('Mapa de Calor de Correlación entre Métricas', fontsize=16, fontweight='bold')
plt.show()

# CONCLUSIÓN ESPERADA:
# Los cuadrados rojo oscuro (cerca de 1) o azul oscuro (cerca de -1) indican métricas muy correlacionadas. 
# Podrías plantearte eliminar una de las dos en la fase de preprocesamiento.